In [1]:
# Metadata for the academic handbook is produced in scraping.ipynb (clause, part, section).
# The next cell loads data/processed/academic_handbook_2025_26_chunks.json — no regex extraction here.

In [2]:
import json
from pathlib import Path

# Academic handbook (from scraping.ipynb): list of { "text", "metadata" }
chunks_path = Path("data/processed/academic_handbook_2025_26_chunks.json")
with open(chunks_path, "r", encoding="utf-8") as f:
    records = json.load(f)

chunks = [r["text"] for r in records]
metadatas = []
for r in records:
    m = dict(r["metadata"])
    for key in list(m.keys()):
        v = m[key]
        if v is None:
            m[key] = ""
        elif isinstance(v, bool):
            pass
        else:
            m[key] = str(v)
    metadatas.append(m)


In [3]:
import chromadb
from chromadb.utils import embedding_functions
import numpy as np

# Must match len(chunks) — generate with upgrade_embeddings.ipynb (intfloat/e5-base-v2)
embeddings = np.load("data/embeddings/academic_handbook_2025_26_e5_base_v2.npy")

# Query-time embeddings must use the same model as indexed vectors
embed_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="intfloat/e5-base-v2"
)
chroma_client = chromadb.PersistentClient(path="data/chroma_db_handbook")

try:
    chroma_client.delete_collection("academic_handbook_2025_26")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="academic_handbook_2025_26",
    embedding_function=embed_func,
)

ids = [f"chunk_{i}" for i in range(len(chunks))]

collection.add(
    ids=ids,
    documents=chunks,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print(f"✅ Rebuilt with metadata: {len(chunks)} chunks")


c:\Users\Hasan Jaafar\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10430.59it/s]
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


✅ Rebuilt with metadata: 839 chunks


In [4]:
def retrieve_with_meta(query: str, top_k: int = 3):
    res = collection.query(
        query_texts=[query],
        n_results=top_k,
        include=["documents", "distances", "metadatas"]  # 👈 added metadatas
    )
    docs  = res["documents"][0]
    dists = res["distances"][0]
    metas = res["metadatas"][0]
    sims  = [1 - d for d in dists]
    return list(zip(sims, metas, docs))

query = "mitigating circumstances academic appeal"
hits = retrieve_with_meta(query, 3)
for i, (sim, meta, doc) in enumerate(hits, 1):
    print(f"\n🔹 Hit {i} | sim≈{sim:.3f} | clause {meta.get('clause')} | part {meta.get('part')} | sec {meta.get('section')}")
    print(doc[:400])


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



🔹 Hit 1 | sim≈0.817 | clause 16.29 | part 4 | sec 16
16.29. Mitigating circumstances will not be considered as grounds for an academic appeal. Any student wishing to have mitigating circumstances considered in respect of an assessment
following the decision of a Progression and Award Board on that assessment should refer to
the University’s Mitigating Circumstances Regulations (Section 11 of the Academic
Regulations).

🔹 Hit 2 | sim≈0.804 | clause 11.49 | part 3 | sec 11
11.49. Where a student is dissatisfied with the outcome of a mitigating circumstances claim, they have a right to submit an academic appeal. The only ground upon which such an appeal can be
made is that there has been a material irregularity in the conduct of the mitigating
circumstances process (refer to the University’s Academic Appeal Regulations for further
information).

🔹 Hit 3 | sim≈0.752 | clause 11.21 | part 3 | sec 11
11.21. There are three possible outcomes to a successful mitigating circumstances claim:
a)

In [5]:
def build_context_block_with_meta(hits):
    lines = []
    for sim, meta, doc in hits:
        cl = meta.get("clause") or "?"
        tag = f"Clause {cl}"
        snippet = doc.strip()
        if len(snippet) > 1200:
            snippet = snippet[:1200] + "…"
        lines.append(f"[{tag}] {snippet}")
    return "\n\n".join(lines)

ctx = build_context_block_with_meta(hits)
print(ctx[:1000])


[Clause 16.29] 16.29. Mitigating circumstances will not be considered as grounds for an academic appeal. Any student wishing to have mitigating circumstances considered in respect of an assessment
following the decision of a Progression and Award Board on that assessment should refer to
the University’s Mitigating Circumstances Regulations (Section 11 of the Academic
Regulations).

[Clause 11.49] 11.49. Where a student is dissatisfied with the outcome of a mitigating circumstances claim, they have a right to submit an academic appeal. The only ground upon which such an appeal can be
made is that there has been a material irregularity in the conduct of the mitigating
circumstances process (refer to the University’s Academic Appeal Regulations for further
information).

[Clause 11.21] 11.21. There are three possible outcomes to a successful mitigating circumstances claim:
a) Extension, or
b) Deferral of an assessment component, or
c) Deferral of a module
